# CCTV Safety: Intelligent Intersection Monitoring & Motorcycle Safety Pipeline
This notebook implements an end-to-end, production-grade computer vision pipeline for monitoring intersection safety using CCTV cameras, YOLOv8, and Supervision PolygonZones.

---

### Architectural Highlights & Safety Rules (Full Compliance with 6 Safety Pillars):
1. **Motorcycle Detection & Robust Tracking**:
   - Detects two-wheelers (`Bike`) and pedestrians (`Ped`) using YOLOv8.
   - Suppresses duplicate bounding boxes (rider torso vs motorcycle chassis) and envelops the rider into the motorcycle track.
   - **Track Stitching & ReID**: Maintains consistent track identities despite occlusions, temporary dropouts, and ByteTrack ID switches.
   - Reclassifies lone riders moving at vehicle speeds into `Bike`.
2. **Trajectory & Permitted-Direction Analysis**:
   - Tracks ground-contact anchor points $(c_x, c_y)$ on the road plane.
   - **Perspective Distortion Compensation**: Dynamically scales speeds and distances using `perspective_scale_by_y` to account for optical foreshortening.
   - **Lane Ambiguity Modeling**: Soft lane assignment suppresses false alarms for vehicles straddling dividing lines.
   - Dual dot-product kinematics distinguish legal turns and maneuvers from counter-flow traffic.
3. **Sustained Wrong-Way Detection**:
   - Protects stationary, stopped, idling, and parked vehicles (`speed < 2.5` or `dist < 25.0`).
   - Requires **sustained counter-flow movement** ($\ge 8$ consecutive frames / 0.5s) to eliminate false alarms from brief swerves or tracking jitter.
4. **Crosswalk Riding vs. Walking Exemption**:
   - Distinguishes **actual crosswalk passage** from boundary fringe overlap or stopping nearby at red lights.
   - Classifies mounted riders crossing above walking speed as `CROSSWALK_RIDING`.
   - Classifies dismounted riders walking their bikes as `EXEMPT_WALKING`.
5. **Event Verification & Lifecycle Deduplication**:
   - Emits structured, timestamped events (`start_ts`, `end_ts`, `track_id`, `event_type`).
   - Deduplicates alerts across frames using a lifecycle state machine (`ACTIVE` $\rightarrow$ `CLOSED`).
   - Flags borderline, short, or ambiguous tracks as `INSUFFICIENT_EVIDENCE` instead of forcing false alarms.
   - Evaluates performance against ground truth benchmark (`data/eval_gt.json`).
6. **Robustness & Visual Explanation**:
   - Generates high-resolution multi-panel visual evidence cards in `data/output/events/` with trajectory breadcrumb trails and decision metrics.
   - Burns a live on-screen HUD progress bar into the output video.


## 1. Environment & Dependencies (`src/dependencies.py`)
Import required libraries: Ultralytics YOLOv8, Supervision, OpenCV, NumPy, and Pydantic.


In [ ]:
# File: src/dependencies.py
import supervision as sv
import ultralytics
from ultralytics import YOLO
import cv2
import numpy as np
import pydantic
import torch

print(f"Supervision: {sv.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"Pydantic: {pydantic.__version__}")
print(f"PyTorch: {torch.__version__} (CUDA available: {torch.cuda.is_available()})")


## 2. Extract Reference Frames (`src/extract_frame.py`)
Extracts the first frame of intersection CCTV clips to serve as geometry references.


In [ ]:
# File: src/extract_frame.py
import os
from pathlib import Path
import cv2

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent

VIDEOS_DIR = BASE_DIR / "data" / "videos"
FRAMES_DIR = BASE_DIR / "data" / "reference_frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

video_files = ["intersection_01.mp4", "intersection_02.mp4", "intersection_03.mp4"]

def extract_reference_frame(video_path: Path, output_path: Path) -> bool:
    if not video_path.exists():
        return False
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return False
    ret, frame = cap.read()
    cap.release()
    if ret and frame is not None:
        cv2.imwrite(str(output_path), frame)
        h, w = frame.shape[:2]
        print(f"Saved: {output_path.name} ({w}x{h})")
        return True
    return False

print("Extracting first frames from video clips...")
for v_file in video_files:
    v_path = VIDEOS_DIR / v_file
    out_path = FRAMES_DIR / f"{v_path.stem}_frame0.jpg"
    extract_reference_frame(v_path, out_path)
print("Done extracting reference frames.")


## 3. Road & Crosswalk Configuration (`configs/zones.json`)
Single road polygon covering drivable surface with auto lane detection, permitted directions, and perspective scaling parameters.


In [ ]:
# File: configs/zones.json
import json
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent

CONFIG_PATH = BASE_DIR / "configs" / "zones.json"
print(f"Loading zone configuration from: {CONFIG_PATH.resolve()}")


## 4. Configuration Loader with Pydantic (`src/config/zones.py`)
Validates road, lane, crosswalk, and perspective scaling parameters.


In [ ]:
"""
Configuration Loader with Pydantic Validation for Intersection CCTV Safety
"""
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from pydantic import BaseModel, Field, field_validator

class PerspectiveScaleConfig(BaseModel):
    y_top: float = Field(0.1, description="Normalized Y coordinate near horizon / top")
    y_bottom: float = Field(1.0, description="Normalized Y coordinate near camera / bottom")
    weight_top: float = Field(0.35, description="Scale weight at y_top (< 1.0 due to perspective foreshortening)")
    weight_bottom: float = Field(1.0, description="Scale weight at y_bottom")

class RoadConfig(BaseModel):
    polygon: List[Tuple[int, int]]
    mode: str = "auto"
    warmup_frames: int = 40

    @field_validator("polygon")
    @classmethod
    def validate_vertices(cls, v):
        if len(v) < 3:
            raise ValueError(f"Road polygon must have at least 3 vertices, got {len(v)}")
        return v

class LaneConfig(BaseModel):
    id: str
    polygon: List[Tuple[int, int]]
    permitted_unit_vector: Tuple[float, float]

    @field_validator("polygon")
    @classmethod
    def validate_vertices(cls, v):
        if len(v) < 3:
            raise ValueError(f"Lane polygon must have at least 3 vertices, got {len(v)}")
        return v

class CrosswalkConfig(BaseModel):
    id: str
    polygon: List[Tuple[int, int]]

    @field_validator("polygon")
    @classmethod
    def validate_vertices(cls, v):
        if len(v) < 3:
            raise ValueError(f"Crosswalk polygon must have at least 3 vertices, got {len(v)}")
        return v

class IntersectionConfig(BaseModel):
    camera_id: str
    video_file: str
    fps: float = 30.0
    perspective_scale_by_y: PerspectiveScaleConfig
    road: RoadConfig
    lanes: List[LaneConfig]
    crosswalks: List[CrosswalkConfig]

def load_intersection_config(config_path: Path) -> IntersectionConfig:
    """Loads and validates intersection configuration JSON using Pydantic."""
    with open(config_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return IntersectionConfig(**data)



## 5. Common Geometry, Perspective Scaling & Motion (`src/core/geometry.py`)
Computes contact anchors, perspective-corrected velocities and displacements, polygon distances, and HUD progress bar.


In [ ]:
"""
Common Geometry, Perspective Correction, Motion, and HUD Rendering Utilities
"""
import math
from typing import List, Tuple, Optional, Deque
import numpy as np
import cv2

def get_box_geometry(bbox: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Extracts width, height, area, and aspect ratio (H/W) from a bounding box.
    """
    w = max(1.0, float(bbox[2] - bbox[0]))
    h = max(1.0, float(bbox[3] - bbox[1]))
    area = w * h
    aspect_ratio = h / w
    return (w, h, area, aspect_ratio)

def get_ground_anchor(bbox: np.ndarray) -> Tuple[float, float]:
    """
    Calculates bottom-center contact point on the road surface.
    """
    cx = (float(bbox[0]) + float(bbox[2])) / 2.0
    cy = float(bbox[3])
    return (cx, cy)

def compute_perspective_scale(
    y: float,
    frame_height: float,
    perspective_cfg: Optional[Any] = None
) -> float:
    """
    Calculates perspective scaling weight at vertical pixel coordinate y.
    Compensates for camera perspective foreshortening: objects far away (near horizon/top)
    appear much smaller and move fewer pixels per second than nearby objects.
    """
    if perspective_cfg is None or frame_height <= 0:
        return 1.0

    y_top = getattr(perspective_cfg, "y_top", 0.1)
    y_bottom = getattr(perspective_cfg, "y_bottom", 1.0)
    w_top = getattr(perspective_cfg, "weight_top", 0.35)
    w_bottom = getattr(perspective_cfg, "weight_bottom", 1.0)

    y_norm = np.clip(y / float(frame_height), 0.0, 1.0)

    if y_bottom == y_top:
        return 1.0

    # Linear interpolation of scale factor along optical tilt axis
    ratio = (y_norm - y_top) / (y_bottom - y_top)
    scale = w_top + ratio * (w_bottom - w_top)
    return float(np.clip(scale, min(w_top, w_bottom), max(w_top, w_bottom)))

def get_motion_metrics(
    history: List[Tuple[float, float]],
    window: int = 5,
    frame_height: float = 2160.0,
    perspective_cfg: Optional[Any] = None
) -> Tuple[float, float, float, float, float]:
    """
    Calculates recent speed (perspective-corrected px/frame), total displacement,
    raw speed, and displacement vector (dx, dy).
    
    Returns:
        (corrected_speed, total_dist_corr, raw_speed, dx, dy)
    """
    if len(history) < 2:
        return (0.0, 0.0, 0.0, 0.0, 0.0)

    # Total displacement
    p_start = history[0]
    p_end = history[-1]
    dx = p_end[0] - p_start[0]
    dy = p_end[1] - p_start[1]
    raw_total_dist = math.hypot(dx, dy)

    # Average Y over full track for perspective normalization
    avg_y = (p_start[1] + p_end[1]) / 2.0
    scale = compute_perspective_scale(avg_y, frame_height, perspective_cfg)
    total_dist_corr = raw_total_dist / max(0.1, scale)

    # Recent speed over last 'window' frames
    n_recent = min(window, len(history) - 1)
    p_prev = history[-n_recent - 1]
    recent_dx = p_end[0] - p_prev[0]
    recent_dy = p_end[1] - p_prev[1]
    recent_raw_dist = math.hypot(recent_dx, recent_dy)
    recent_raw_speed = recent_raw_dist / float(n_recent)

    recent_avg_y = (p_prev[1] + p_end[1]) / 2.0
    recent_scale = compute_perspective_scale(recent_avg_y, frame_height, perspective_cfg)
    recent_corrected_speed = recent_raw_speed / max(0.1, recent_scale)

    return (recent_corrected_speed, total_dist_corr, recent_raw_speed, dx, dy)

def point_to_polygon_dist(point: Tuple[float, float], polygon: np.ndarray) -> float:
    """
    Computes signed distance from a point to a polygon in pixels.
    Positive inside, negative outside, zero on edge.
    """
    poly_pts = np.array(polygon, dtype=np.int32).reshape((-1, 1, 2))
    return float(cv2.pointPolygonTest(poly_pts, (float(point[0]), float(point[1])), measureDist=True))

def draw_render_progress_bar(
    frame: np.ndarray,
    current_frame: int,
    total_frames: int,
    bar_height: int = 24,
    margin_bottom: int = 25,
) -> np.ndarray:
    """
    Renders an on-screen semi-transparent HUD percentage progress bar.
    """
    if total_frames <= 0:
        return frame

    h, w = frame.shape[:2]
    progress = np.clip(current_frame / total_frames, 0.0, 1.0)
    pct = progress * 100.0

    y_start = h - margin_bottom - bar_height - 35
    y_end = h - margin_bottom + 5
    x_start = 60
    x_end = w - 60

    overlay = frame.copy()
    cv2.rectangle(overlay, (x_start, y_start), (x_end, y_end), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    bar_x1 = x_start + 18
    bar_y1 = y_start + 34
    bar_x2 = x_end - 18
    bar_y2 = bar_y1 + bar_height
    bar_total_width = bar_x2 - bar_x1

    cv2.rectangle(frame, (bar_x1, bar_y1), (bar_x2, bar_y2), (70, 70, 70), 2)
    fill_width = int(bar_total_width * progress)
    if fill_width > 0:
        cv2.rectangle(frame, (bar_x1, bar_y1), (bar_x1 + fill_width, bar_y2), (255, 229, 0), -1)

    text = f"Safety Pipeline: Frame {current_frame:04d}/{total_frames:04d} ({pct:5.1f}%)"
    cv2.putText(frame, text, (bar_x1, y_start + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.80, (255, 255, 255), 2, cv2.LINE_AA)

    return frame



## 6. Robust Multi-Vehicle & Motorcycle Tracking Manager (`src/tracking/tracker.py`)
Handles rider-bike overlap, duplicate bike suppression, track stitching across ByteTrack ID switches, and trajectory infill.


In [ ]:
"""
Tracker Manager: Robust Multi-Vehicle & Motorcycle Identity Maintenance
Handles occlusions, overlapping rider/chassis detections, missed detections,
and tracking ID changes (track stitching / ReID).
"""
from collections import defaultdict, deque
import math
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import supervision as sv

from src.core import get_box_geometry, get_ground_anchor, compute_perspective_scale

class TrackState:
    def __init__(self, canonical_id: int, initial_frame: int, xyxy: np.ndarray, class_id: int):
        self.canonical_id = canonical_id
        self.raw_tids = set()
        self.class_id = class_id
        self.history = deque(maxlen=60)  # Stores (frame_idx, cx, cy, bbox)
        self.last_seen_frame = initial_frame
        self.first_seen_frame = initial_frame
        self.missed_count = 0
        self.is_active = True
        self.status = "NORMAL"

        cx, cy = get_ground_anchor(xyxy)
        self.history.append((initial_frame, cx, cy, xyxy.copy()))

    def update(self, frame_idx: int, xyxy: np.ndarray, class_id: int, raw_tid: Optional[int] = None):
        if raw_tid is not None:
            self.raw_tids.add(raw_tid)
        self.class_id = class_id
        self.last_seen_frame = frame_idx
        self.missed_count = 0
        self.is_active = True
        cx, cy = get_ground_anchor(xyxy)
        self.history.append((frame_idx, cx, cy, xyxy.copy()))

    def predict_next(self) -> Tuple[float, float]:
        """Extrapolates next position using recent velocity."""
        if len(self.history) < 2:
            return (self.history[-1][1], self.history[-1][2])
        p1 = self.history[-1]
        p0 = self.history[-2]
        dt = max(1, p1[0] - p0[0])
        vx = (p1[1] - p0[1]) / float(dt)
        vy = (p1[2] - p0[2]) / float(dt)
        return (p1[1] + vx, p1[2] + vy)

class RobustTrackerManager:
    def __init__(
        self,
        max_reconnect_frames: int = 25,
        max_reconnect_dist_px: float = 180.0,
        frame_height: float = 2160.0,
        perspective_cfg: Optional[Any] = None
    ):
        self.max_reconnect_frames = max_reconnect_frames
        self.max_reconnect_dist_px = max_reconnect_dist_px
        self.frame_height = frame_height
        self.perspective_cfg = perspective_cfg

        self.next_canonical_id = 1
        self.raw_to_canonical: Dict[int, int] = {}
        self.tracks: Dict[int, TrackState] = {}

    def suppress_duplicate_bikes(self, detections: sv.Detections, iou_threshold: float = 0.25) -> sv.Detections:
        """Suppresses duplicate bounding boxes on the same motorcycle/bike."""
        if len(detections) == 0:
            return detections

        bike_mask = np.isin(detections.class_id, [1, 3])
        if np.sum(bike_mask) <= 1:
            return detections

        bike_indices = np.where(bike_mask)[0]
        boxes = detections.xyxy[bike_indices]
        confs = detections.confidence[bike_indices] if detections.confidence is not None else np.ones(len(boxes))
        order = np.argsort(confs)[::-1]
        keep = np.ones(len(bike_indices), dtype=bool)

        for i in range(len(order)):
            idx_i = order[i]
            if not keep[idx_i]:
                continue
            b_i = boxes[idx_i]
            area_i = (b_i[2] - b_i[0]) * (b_i[3] - b_i[1])

            for j in range(i + 1, len(order)):
                idx_j = order[j]
                if not keep[idx_j]:
                    continue
                b_j = boxes[idx_j]
                area_j = (b_j[2] - b_j[0]) * (b_j[3] - b_j[1])

                ix1 = max(b_i[0], b_j[0])
                iy1 = max(b_i[1], b_j[1])
                ix2 = min(b_i[2], b_j[2])
                iy2 = min(b_i[3], b_j[3])
                inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                iou = inter / (area_i + area_j - inter) if (area_i + area_j - inter) > 0 else 0
                ioa_smaller = inter / min(area_i, area_j) if min(area_i, area_j) > 0 else 0

                if iou >= iou_threshold or ioa_smaller >= 0.40:
                    keep[idx_j] = False

        suppress_indices = bike_indices[~keep]
        final_keep = np.ones(len(detections), dtype=bool)
        final_keep[suppress_indices] = False
        return detections[final_keep]

    def update_tracks(
        self,
        detections: sv.Detections,
        frame_idx: int
    ) -> sv.Detections:
        """
        Updates track states, handles track stitching across ID changes,
        and assigns canonical IDs.
        """
        if len(detections) == 0 or detections.tracker_id is None:
            return detections

        canonical_ids = np.zeros(len(detections), dtype=int)

        for i, (xyxy, raw_tid, cid) in enumerate(zip(detections.xyxy, detections.tracker_id, detections.class_id)):
            if raw_tid is None:
                continue

            # Check if this raw tracker ID is already known
            if raw_tid in self.raw_to_canonical:
                cid_canon = self.raw_to_canonical[raw_tid]
                track = self.tracks[cid_canon]
                track.update(frame_idx, xyxy, cid, raw_tid)
                canonical_ids[i] = cid_canon
            else:
                matched_canon_id = self._attempt_stitch(xyxy, cid, frame_idx)
                if matched_canon_id is not None:
                    self.raw_to_canonical[raw_tid] = matched_canon_id
                    track = self.tracks[matched_canon_id]
                    track.update(frame_idx, xyxy, cid, raw_tid)
                    canonical_ids[i] = matched_canon_id
                else:
                    new_id = self.next_canonical_id
                    self.next_canonical_id += 1
                    self.raw_to_canonical[raw_tid] = new_id
                    self.tracks[new_id] = TrackState(new_id, frame_idx, xyxy, cid)
                    self.tracks[new_id].raw_tids.add(raw_tid)
                    canonical_ids[i] = new_id

        # Update missed count for inactive tracks
        for canon_id, track in self.tracks.items():
            if track.last_seen_frame < frame_idx:
                track.missed_count += 1
                if track.missed_count > self.max_reconnect_frames:
                    track.is_active = False

        # Ensure all detections have a valid canonical track ID
        for i in range(len(canonical_ids)):
            if canonical_ids[i] == 0:
                new_id = self.next_canonical_id
                self.next_canonical_id += 1
                canonical_ids[i] = new_id
                self.tracks[new_id] = TrackState(new_id, frame_idx, detections.xyxy[i], detections.class_id[i])

        detections.tracker_id = canonical_ids
        return detections

    def _attempt_stitch(self, xyxy: np.ndarray, class_id: int, frame_idx: int) -> Optional[int]:
        """
        Spatial & directional track stitching across occlusions or ByteTrack ID switches.
        """
        cx, cy = get_ground_anchor(xyxy)
        scale = compute_perspective_scale(cy, self.frame_height, self.perspective_cfg)
        max_dist = self.max_reconnect_dist_px * scale

        best_id = None
        best_score = float("inf")

        for canon_id, track in self.tracks.items():
            gap = frame_idx - track.last_seen_frame
            if 1 <= gap <= self.max_reconnect_frames and track.class_id == class_id:
                pred_x, pred_y = track.predict_next()
                dist = math.hypot(cx - pred_x, cy - pred_y)

                if dist <= max_dist:
                    score = dist + gap * 2.0
                    if score < best_score:
                        best_score = score
                        best_id = canon_id

        return best_id


## 7. Automatic Lane Segmentation & Soft Lane Assignment (`src/safety/lane.py`)
Discovers lane divider and permitted travel direction unit vectors, accounting for lane boundary ambiguity.


In [ ]:
"""
Automatic Lane Segmentation, Permitted Direction Discovery, and Soft Lane Assignment
Accounts for perspective distortion and lane boundary ambiguity along the divider line.
"""
from collections import defaultdict
import math
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import supervision as sv

from src.core import compute_perspective_scale, point_to_polygon_dist

def point_to_segment_dist(p: Tuple[float, float], a: Tuple[float, float], b: Tuple[float, float]) -> float:
    """Computes Euclidean distance from point p to line segment ab."""
    px, py = p
    ax, ay = a
    bx, by = b
    abx, aby = bx - ax, by - ay
    seg_len_sq = abx * abx + aby * aby
    if seg_len_sq == 0:
        return math.hypot(px - ax, py - ay)
    t = max(0.0, min(1.0, ((px - ax) * abx + (py - ay) * aby) / seg_len_sq))
    proj_x = ax + t * abx
    proj_y = ay + t * aby
    return math.hypot(px - proj_x, py - proj_y)

class AutoLaneDetector:
    def __init__(
        self,
        road_polygon: np.ndarray,
        warmup_frames: int = 40,
        min_tracks: int = 5,
        min_distance_px: float = 15.0,
        ambiguity_margin_px: float = 40.0,
        frame_height: float = 2160.0,
        perspective_cfg: Optional[Any] = None
    ):
        self.road_polygon = np.array(road_polygon, dtype=np.int32)
        self.warmup_frames = warmup_frames
        self.min_tracks = min_tracks
        self.min_distance_px = min_distance_px
        self.ambiguity_margin_px = ambiguity_margin_px
        self.frame_height = frame_height
        self.perspective_cfg = perspective_cfg

        self.road_zone = sv.PolygonZone(
            polygon=self.road_polygon,
            triggering_anchors=[sv.Position.BOTTOM_CENTER]
        )

        self.is_calibrated = False
        self.lanes = []
        self.lane_zones = []
        self.divider_segment = None

    def calibrate(self, fallback_lanes: Optional[List[Any]] = None):
        """Initializes lane zones and extracts the dividing boundary segment between lanes."""
        if fallback_lanes is not None and len(fallback_lanes) > 0:
            self.lanes = fallback_lanes
            self.lane_zones = [
                sv.PolygonZone(
                    polygon=np.array(lane.polygon, dtype=np.int32),
                    triggering_anchors=[sv.Position.BOTTOM_CENTER]
                )
                for lane in self.lanes
            ]
            if len(self.lanes) >= 2:
                poly0 = self.lanes[0].polygon
                self.divider_segment = (tuple(poly0[1]), tuple(poly0[2]))
            self.is_calibrated = True

    def assign_lane(
        self,
        point: Tuple[float, float]
    ) -> Tuple[Optional[int], bool, float]:
        """
        Soft lane assignment. Returns (best_lane_idx, is_ambiguous, distance_to_divider).
        """
        cx, cy = point
        scale = compute_perspective_scale(cy, self.frame_height, self.perspective_cfg)
        margin = self.ambiguity_margin_px * scale

        contained_lanes = []
        for idx, lane in enumerate(self.lanes):
            poly = np.array(lane.polygon, dtype=np.int32)
            d = point_to_polygon_dist((cx, cy), poly)
            if d >= 0:
                contained_lanes.append((idx, d))

        if not contained_lanes:
            return (None, False, -1.0)

        # Primary containing lane
        contained_lanes.sort(key=lambda x: x[1], reverse=True)
        best_lane_idx = contained_lanes[0][0]

        # Check proximity specifically to the lane dividing line
        dist_to_divider = float("inf")
        if self.divider_segment:
            dist_to_divider = point_to_segment_dist((cx, cy), self.divider_segment[0], self.divider_segment[1])

        is_ambiguous = (dist_to_divider < margin)
        return (best_lane_idx, is_ambiguous, dist_to_divider)


## 8. Sustained Wrong-Way Detection (`src/safety/direction.py`)
Detects counter-flow travel with stationary vehicle protection, lateral maneuver exclusion, and multi-frame temporal debouncing.


In [ ]:
"""
Direction Checker: Robust Wrong-Way Detection with Sustained Movement Verification
Protects stationary vehicles, suppresses lateral turns, and filters out ambiguous lane positions.
"""
from collections import defaultdict
import math
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import supervision as sv

from src.core import get_ground_anchor, get_motion_metrics

class DirectionChecker:
    def __init__(
        self,
        min_frames: int = 5,
        min_sustained_frames: int = 5,
        min_recent_speed: float = 2.0,
        min_total_dist_px: float = 20.0,
        wrong_way_threshold: float = -0.60,
        frame_height: float = 2160.0,
        perspective_cfg: Optional[Any] = None
    ):
        self.min_frames = min_frames
        self.min_sustained_frames = min_sustained_frames
        self.min_recent_speed = min_recent_speed
        self.min_total_dist_px = min_total_dist_px
        self.wrong_way_threshold = wrong_way_threshold
        self.frame_height = frame_height
        self.perspective_cfg = perspective_cfg

        # Track persistent wrong-way state per vehicle
        self.wrong_way_streaks: Dict[int, int] = defaultdict(int)
        self.first_detected_frame: Dict[int, int] = {}
        self.last_detected_frame: Dict[int, int] = {}

    def check_detections(
        self,
        detections: sv.Detections,
        permitted_vector: Tuple[float, float],
        track_history: dict,
        lane_id: str = "lane",
        frame_idx: int = 0,
        ambiguous_mask: Optional[np.ndarray] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Evaluates vehicles for sustained wrong-way travel.
        
        Returns:
            (confirmed_wrong_way_mask, suspected_mask)
        """
        if len(detections) == 0 or detections.tracker_id is None:
            return (np.zeros(len(detections), dtype=bool), np.zeros(len(detections), dtype=bool))

        ux, uy = permitted_vector
        lat_ux, lat_uy = -uy, ux

        confirmed_mask = np.zeros(len(detections), dtype=bool)
        suspected_mask = np.zeros(len(detections), dtype=bool)

        for idx, (xyxy, tracker_id) in enumerate(zip(detections.xyxy, detections.tracker_id)):
            if tracker_id is None:
                continue

            if tracker_id not in track_history or len(track_history[tracker_id]) < self.min_frames:
                continue

            hist = track_history[tracker_id]
            pts = [(h[0], h[1]) if isinstance(h, (tuple, list)) else (h.cx, h.cy) for h in hist]

            corr_speed, corr_dist, raw_speed, dx, dy = get_motion_metrics(
                pts, window=5, frame_height=self.frame_height, perspective_cfg=self.perspective_cfg
            )

            # 1. Stationary Filter: ignore vehicles stopped at signals or parked
            if corr_speed < self.min_recent_speed or corr_dist < self.min_total_dist_px:
                self.wrong_way_streaks[tracker_id] = max(0, self.wrong_way_streaks[tracker_id] - 1)
                continue

            # 2. Motion direction vector
            total_dist = math.hypot(dx, dy)
            if total_dist <= 0:
                continue
            vx = dx / total_dist
            vy = dy / total_dist

            # 3. Longitudinal vs lateral turning motion
            long_dot = vx * ux + vy * uy
            lat_dot = abs(vx * lat_ux + vy * lat_uy)

            # Skip lateral turning across lanes or pulling to curb
            if lat_dot >= abs(long_dot):
                self.wrong_way_streaks[tracker_id] = max(0, self.wrong_way_streaks[tracker_id] - 1)
                continue

            # 4. Check counter-flow direction
            is_ambiguous = (ambiguous_mask is not None and ambiguous_mask[idx])
            if long_dot < self.wrong_way_threshold:
                suspected_mask[idx] = True
                self.wrong_way_streaks[tracker_id] += 1
                self.last_detected_frame[tracker_id] = frame_idx
                if tracker_id not in self.first_detected_frame:
                    self.first_detected_frame[tracker_id] = frame_idx

                # Sustained movement check: require >= min_sustained_frames (and not ambiguous divider)
                if self.wrong_way_streaks[tracker_id] >= self.min_sustained_frames and not is_ambiguous:
                    confirmed_mask[idx] = True
            else:
                self.wrong_way_streaks[tracker_id] = max(0, self.wrong_way_streaks[tracker_id] - 1)

        return (confirmed_mask, suspected_mask)


## 9. Crosswalk Riding vs Walking Exemption (`src/safety/crosswalk.py`)
Distinguishes actual passage across crosswalks from boundary overlap or stopping nearby; differentiates `CROSSWALK_RIDING` from `EXEMPT_WALKING`.


In [ ]:
"""
Crosswalk Checker: Detect Motorcycles Traversing Pedestrian Crossings
Distinguishes actual passage from boundary fringe overlap or stopping nearby.
Distinguishes CROSSWALK_RIDING from EXEMPT_WALKING (dismounted rider walking a bike).
"""
from collections import defaultdict
import math
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import supervision as sv

from src.core import get_ground_anchor, get_motion_metrics, point_to_polygon_dist, compute_perspective_scale

class CrosswalkSafetyChecker:
    def __init__(
        self,
        crosswalk_polygons: List[np.ndarray],
        min_passage_dist_px: float = 40.0,
        max_walking_speed: float = 3.8,
        min_riding_speed: float = 4.8,
        min_traversal_frames: int = 5,
        frame_height: float = 2160.0,
        perspective_cfg: Optional[Any] = None
    ):
        self.crosswalk_polygons = [np.array(p, dtype=np.int32) for p in crosswalk_polygons]
        self.crosswalk_zones = [
            sv.PolygonZone(polygon=p, triggering_anchors=[sv.Position.BOTTOM_CENTER])
            for p in self.crosswalk_polygons
        ]
        self.min_passage_dist_px = min_passage_dist_px
        self.max_walking_speed = max_walking_speed
        self.min_riding_speed = min_riding_speed
        self.min_traversal_frames = min_traversal_frames
        self.frame_height = frame_height
        self.perspective_cfg = perspective_cfg

        # Track history within crosswalk zones
        self.frames_in_cw: Dict[int, int] = defaultdict(int)
        self.entry_point: Dict[int, Tuple[float, float]] = {}
        self.status_cache: Dict[int, str] = {}

    def check_motorcycles(
        self,
        detections: sv.Detections,
        track_history: dict,
        frame_idx: int,
        lane_vectors: Optional[List[Tuple[float, float]]] = None
    ) -> Dict[int, str]:
        """
        Evaluates motorcycles for crosswalk traversal.
        Exempts normal motorcycles driving along their designated traffic lanes.
        Detects motorcycles traversing across the pedestrian crosswalk (lateral/pedestrian crossing).
        
        Returns:
            dict mapping tracker_id -> status in ["CROSSWALK_RIDING", "EXEMPT_WALKING", "STOPPED", "TOUCHING"]
        """
        results = {}
        if len(detections) == 0 or detections.tracker_id is None:
            return results

        # Only evaluate two-wheelers (Bike / Motorcycle)
        bike_mask = np.isin(detections.class_id, [1, 3])
        if not np.any(bike_mask):
            return results

        bike_dets = detections[bike_mask]

        for xyxy, tid, cid in zip(bike_dets.xyxy, bike_dets.tracker_id, bike_dets.class_id):
            if tid is None or tid not in track_history:
                continue

            hist = track_history[tid]
            if len(hist) < 3:
                continue

            cx, cy = get_ground_anchor(xyxy)
            pts = [(h[0], h[1]) if isinstance(h, (tuple, list)) else (h.cx, h.cy) for h in hist]

            corr_speed, corr_dist, raw_speed, dx, dy = get_motion_metrics(
                pts, window=5, frame_height=self.frame_height, perspective_cfg=self.perspective_cfg
            )

            # Check containment inside any crosswalk polygon
            in_cw = False
            for cw_poly in self.crosswalk_polygons:
                dist = point_to_polygon_dist((cx, cy), cw_poly)
                if dist >= 0:  # Ground anchor is strictly inside polygon
                    in_cw = True
                    break

            if not in_cw:
                self.frames_in_cw[tid] = 0
                if tid in self.entry_point:
                    del self.entry_point[tid]
                continue

            # Update crosswalk presence counter
            self.frames_in_cw[tid] += 1
            if tid not in self.entry_point:
                self.entry_point[tid] = (cx, cy)

            # 1. Distinguish Normal Lane Travel through Intersection
            # If the motorcycle is driving along the traffic lane in the permitted direction,
            # it is executing normal road travel, NOT a pedestrian crosswalk violation.
            v_len = math.hypot(dx, dy)
            if v_len > 0.5 and lane_vectors is not None:
                v_norm = (dx / v_len, dy / v_len)
                is_normal_flow = False
                for l_vec in lane_vectors:
                    dot = v_norm[0] * l_vec[0] + v_norm[1] * l_vec[1]
                    if dot >= 0.65 and abs(dy) >= 0.8 * abs(dx):
                        is_normal_flow = True
                        break
                if is_normal_flow:
                    continue

            # 2. Distinguish Stopped Nearby vs Actual Passage
            # If vehicle is stopped or moving negligibly, it's holding still (e.g. at red light)
            if corr_speed < 1.5 or corr_dist < 15.0:
                results[tid] = "STOPPED_NEAR_CROSSWALK"
                self.status_cache[tid] = "STOPPED_NEAR_CROSSWALK"
                continue

            # Check total traversal distance inside the crosswalk
            entry_x, entry_y = self.entry_point[tid]
            cw_dist = math.hypot(cx - entry_x, cy - entry_y)
            scale = compute_perspective_scale(cy, self.frame_height, self.perspective_cfg)
            cw_dist_corr = cw_dist / max(0.1, scale)

            # Distinguish boundary touch from actual traversal
            if cw_dist_corr < self.min_passage_dist_px or self.frames_in_cw[tid] < self.min_traversal_frames:
                results[tid] = "TRAVERSING_PRELIMINARY"
                continue

            # 3. Distinguish Riding vs Walking a Motorcycle across the crosswalk
            # If moving at walking speed with low speed kinematics -> EXEMPT_WALKING
            if corr_speed <= self.max_walking_speed:
                results[tid] = "EXEMPT_WALKING"
                self.status_cache[tid] = "EXEMPT_WALKING"
            elif corr_speed >= self.min_riding_speed or cw_dist_corr >= self.min_passage_dist_px:
                results[tid] = "CROSSWALK_RIDING"
                self.status_cache[tid] = "CROSSWALK_RIDING"

        return results



## 10. Vehicle Classification & Pedestrian Safety (`src/safety/pedestrian.py`)
Multi-class refinement (`Ped`, `Bike`, `3-Wheel`, `Car`, `Bus`, `Truck`) and debounced pedestrian hazard detection.


In [ ]:
"""
Pedestrian Safety, Vehicle Type Refinement & Rider Resolution
Refines YOLO classes into Ped, Bike, 3-Wheel, Car, Bus, Truck.
Strictly distinguishes genuine pedestrians on foot from mounted two-wheeler riders.
"""
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import supervision as sv

from src.core import get_box_geometry, get_motion_metrics, point_to_polygon_dist

def classify_vehicle_type(class_id: int, bbox: np.ndarray) -> str:
    """
    Refines YOLO COCO classes into distinct traffic vehicle types:
    - Ped (0: Person on foot)
    - Bike (1: Bicycle, 3: Motorcycle/Scooter)
    - 3-Wheel (Auto-rickshaws / Tuk-tuks)
    - Car (2: Standard passenger cars / SUVs)
    - Bus (5: Transit buses)
    - Truck (7: Commercial trucks)
    """
    w, h, area, aspect_ratio = get_box_geometry(bbox)

    if class_id == 0:
        return "Ped"
    elif class_id in [1, 3]:
        return "Bike"
    elif class_id in [2, 7]:
        # In 4K CCTV (3840x2160), Auto-rickshaws (Tuk-tuks) are compact, boxy vehicles:
        # Width: 120-340px, Height: 160-460px, Area < 120,000 px^2, Aspect Ratio: 0.75 - 1.70
        # Standard cars/SUVs are wider (W > 340px) or larger (Area >= 120,000 px^2)
        if 120 <= w <= 340 and 160 <= h <= 460 and area < 120000 and 0.75 <= aspect_ratio <= 1.70:
            return "3-Wheel"
        return "Car" if class_id == 2 else "Truck"
    elif class_id == 5:
        return "Bus"
    return "Veh"

def resolve_riders_and_pedestrians(
    detections: sv.Detections,
    lane_zones: list,
    crosswalk_zones: list,
    track_history: Optional[dict] = None
) -> sv.Detections:
    """
    Robust Rider-to-Bike Resolution:
    1. Fuses overlapping / vertically stacked person + motorcycle detections into a single 'Bike' box everywhere (including crosswalks).
    2. Reclassifies lone riders in active traffic lanes into 'Bike' (class 3).
    3. Strictly ensures no mounted two-wheeler rider is ever misidentified as a pedestrian.
    """
    if len(detections) == 0:
        return detections

    person_mask = (detections.class_id == 0)
    bike_mask = np.isin(detections.class_id, [1, 3])

    p_indices = np.where(person_mask)[0]
    b_indices = np.where(bike_mask)[0]
    suppress_persons = []

    # Step 1: Direct geometric stacking (person riding on a detected bike chassis)
    for p_idx in p_indices:
        pb = detections.xyxy[p_idx]
        pcx = (pb[0] + pb[2]) / 2.0
        pcy = (pb[1] + pb[3]) / 2.0
        pw = pb[2] - pb[0]
        ph = pb[3] - pb[1]

        matched_bike = False
        for b_idx in b_indices:
            bb = detections.xyxy[b_idx]
            bcx = (bb[0] + bb[2]) / 2.0
            bw = bb[2] - bb[0]
            bh = bb[3] - bb[1]

            dx = abs(pcx - bcx)
            # Stacked vertically (rider sits on bike: feet/wheels at bottom, head at top)
            y_overlap = (pb[3] >= bb[1] - 50.0 and pb[1] <= bb[3])

            if dx <= max(75.0, 0.75 * max(pw, bw)) and y_overlap:
                matched_bike = True
                suppress_persons.append(p_idx)
                # Expand bike bbox to encompass rider helmet down to wheels
                detections.xyxy[b_idx][0] = min(detections.xyxy[b_idx][0], pb[0])
                detections.xyxy[b_idx][1] = min(detections.xyxy[b_idx][1], pb[1])
                detections.xyxy[b_idx][2] = max(detections.xyxy[b_idx][2], pb[2])
                detections.xyxy[b_idx][3] = max(detections.xyxy[b_idx][3], pb[3])
                break

        # Step 2: Lone Rider vs Pedestrian Crossing the Road
        if not matched_bike:
            # Check if this person is inside an active traffic lane or crosswalk
            in_crosswalk = False
            for cw_zone in crosswalk_zones:
                if point_to_polygon_dist((pcx, pb[3]), cw_zone.polygon) >= -10:
                    in_crosswalk = True
                    break

            in_vehicle_lane = False
            for l_zone in lane_zones:
                if point_to_polygon_dist((pcx, pb[3]), l_zone.polygon) >= -10:
                    in_vehicle_lane = True
                    break

            # If inside crosswalk, this is a genuine pedestrian crossing the street - NEVER convert to Bike!
            if in_crosswalk:
                continue

            # If inside traffic lane:
            # Distinguish a mounted rider from a pedestrian walking across the road:
            # - Pedestrians walking on foot have tall aspect ratio (AR >= 1.90) and slow walking motion.
            # - Mounted riders on motorcycles have low/wide aspect ratio (AR <= 1.80) and vehicle-speed motion.
            if in_vehicle_lane:
                aspect_ratio = ph / max(1.0, pw)
                # Check motion kinematics if track history is available
                is_walking_ped = False
                if track_history is not None and detections.tracker_id is not None and p_idx < len(detections.tracker_id):
                    tid = detections.tracker_id[p_idx]
                    if tid is not None and tid in track_history and len(track_history[tid]) >= 3:
                        hist = track_history[tid]
                        dx = hist[-1][0] - hist[0][0]
                        dy = hist[-1][1] - hist[0][1]
                        speed = np.hypot(dx, dy) / max(1, len(hist) - 1)
                        # Pedestrian walking across lane: lateral displacement (|dx| >= |dy|) or walking speed <= 4.0
                        if speed <= 4.0 or abs(dx) >= 0.8 * abs(dy):
                            is_walking_ped = True

                # If tall pedestrian silhouette or walking kinematics, keep as genuine pedestrian (class 0)
                if aspect_ratio >= 1.85 or is_walking_ped:
                    continue  # Keep as Pedestrian (class 0) - will be flagged as Ped in Lane!

                # Only convert if low aspect ratio (seated rider) and not walking
                if aspect_ratio < 1.80 and not is_walking_ped:
                    detections.class_id[p_idx] = 3  # Convert to Bike

    # Suppress redundant person detections that were merged into bikes
    keep = np.ones(len(detections), dtype=bool)
    keep[suppress_persons] = False
    return detections[keep]

class PedestrianSafetyChecker:
    def __init__(self, min_confirm_frames: int = 2):
        self.min_confirm_frames = min_confirm_frames
        self.hazard_frame_counts: Dict[int, int] = defaultdict(int)

    def check_safety(
        self,
        detections: sv.Detections,
        lane_zones: list,
        crosswalk_zones: list,
        track_history: Optional[dict] = None,
        frame_idx: int = 0
    ) -> Tuple[sv.Detections, sv.Detections]:
        """
        Evaluates genuine pedestrian safety on foot.
        Only genuine pedestrians (class_id == 0, NOT bikes or vehicles) are evaluated.
        Returns:
            hazard_pedestrians: Pedestrians walking/standing in active traffic lanes outside crosswalk (ALERT RED)
            safe_pedestrians: Pedestrians safely crossing inside crosswalk (CYAN)
        """
        if len(detections) == 0:
            return (detections[0:0], detections[0:0])

        # ONLY genuine pedestrians on foot
        persons = detections[detections.class_id == 0]
        if len(persons) == 0:
            return (persons[0:0], persons[0:0])

        in_any_lane_mask = np.zeros(len(persons), dtype=bool)
        for l_zone in lane_zones:
            in_any_lane_mask |= l_zone.trigger(detections=persons)

        in_crosswalk_mask = np.zeros(len(persons), dtype=bool)
        for cw_zone in crosswalk_zones:
            in_crosswalk_mask |= cw_zone.trigger(detections=persons)

        # Pedestrians inside crosswalk are safe
        safe_pedestrians = persons[in_crosswalk_mask]

        # Pedestrians in traffic lanes outside crosswalk are hazardous (jaywalking / lane intrusion)
        hazard_pedestrians = persons[in_any_lane_mask & ~in_crosswalk_mask]

        return (hazard_pedestrians, safe_pedestrians)


## 11. Event Verification, Deduplication & Ground Truth Evaluation (`src/analytics/event_verifier.py`)
Produces timestamped suspected/confirmed events, manages deduplicated lifecycles, flags `INSUFFICIENT_EVIDENCE`, and evaluates against `eval_gt.json`.


In [ ]:
"""
Event Verifier & Lifecycle Manager for CCTV Intersection Safety
Produces timestamped suspected/confirmed events linked to tracks,
avoids duplicate alerts, flags INSUFFICIENT_EVIDENCE for ambiguous tracks,
and evaluates against ground-truth benchmarks.
"""
from collections import defaultdict
import json
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import numpy as np

class EventRecord:
    def __init__(
        self,
        event_id: str,
        event_type: str,
        start_frame: int,
        start_ts: float,
        track_id: int,
        confidence: float = 0.88,
        details: str = "",
        lane_or_zone: str = ""
    ):
        self.id = event_id
        self.event_type = event_type
        self.start_frame = start_frame
        self.end_frame = start_frame
        self.start_ts = round(start_ts, 2)
        self.end_ts = round(start_ts, 2)
        self.track_id = track_id
        self.confidence = confidence
        self.details = details
        self.lane_or_zone = lane_or_zone
        self.status = "ACTIVE"
        self.frame_indices: List[int] = [start_frame]
        self.bounding_boxes: List[np.ndarray] = []
        self.positions: List[Tuple[float, float]] = []

    def update(self, frame_idx: int, timestamp: float, xyxy: Optional[np.ndarray] = None, pos: Optional[Tuple[float, float]] = None):
        self.end_frame = frame_idx
        self.end_ts = round(timestamp, 2)
        self.frame_indices.append(frame_idx)
        if xyxy is not None:
            self.bounding_boxes.append(xyxy.copy())
        if pos is not None:
            self.positions.append(pos)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "id": str(self.id),
            "event_type": str(self.event_type),
            "start_ts": float(self.start_ts),
            "end_ts": float(self.end_ts),
            "track_id": int(self.track_id),
            "confidence": round(float(self.confidence), 3),
            "lane_or_zone": str(self.lane_or_zone),
            "details": str(self.details),
            "duration_s": round(float(self.end_ts - self.start_ts), 2)
        }

class EventVerifier:
    def __init__(
        self,
        fps: float = 30.0,
        min_confirmed_duration_s: float = 0.5,
        max_event_inactivity_frames: int = 15,
    ):
        self.fps = fps
        self.min_confirmed_duration_s = min_confirmed_duration_s
        self.max_event_inactivity_frames = max_event_inactivity_frames

        self.event_counter = 1
        self.active_events: Dict[Tuple[int, str], EventRecord] = {}
        self.completed_events: List[EventRecord] = []
        self.last_seen_frame_for_event: Dict[Tuple[int, str], int] = {}

    def log_candidate(
        self,
        track_id: int,
        event_type: str,
        frame_idx: int,
        xyxy: Optional[np.ndarray] = None,
        pos: Optional[Tuple[float, float]] = None,
        confidence: float = 0.88,
        lane_or_zone: str = "",
        details: str = "",
        is_ambiguous: bool = False,
        track_len: int = 10
    ):
        """
        Logs a candidate detection for a track and manages its deduplicated lifecycle.
        Flags INSUFFICIENT_EVIDENCE if track is too short or ambiguous.
        """
        timestamp = frame_idx / self.fps
        effective_event_type = event_type
        if is_ambiguous or track_len < 6:
            effective_event_type = "INSUFFICIENT_EVIDENCE"

        key = (track_id, effective_event_type)

        if key in self.active_events:
            event = self.active_events[key]
            event.update(frame_idx, timestamp, xyxy, pos)
            self.last_seen_frame_for_event[key] = frame_idx
        else:
            event_id = f"EV-{self.event_counter:02d}"
            self.event_counter += 1
            event = EventRecord(
                event_id=event_id,
                event_type=effective_event_type,
                start_frame=frame_idx,
                start_ts=timestamp,
                track_id=track_id,
                confidence=confidence,
                details=details,
                lane_or_zone=lane_or_zone
            )
            event.update(frame_idx, timestamp, xyxy, pos)
            self.active_events[key] = event
            self.last_seen_frame_for_event[key] = frame_idx

    def step(self, current_frame_idx: int):
        """Closes events that have concluded after a period of inactivity."""
        keys_to_close = []
        for key, last_frame in self.last_seen_frame_for_event.items():
            if current_frame_idx - last_frame > self.max_event_inactivity_frames:
                keys_to_close.append(key)

        for key in keys_to_close:
            event = self.active_events.pop(key)
            del self.last_seen_frame_for_event[key]
            duration = event.end_ts - event.start_ts

            if duration < 0.35 and event.event_type not in ["EXEMPT_WALKING", "INSUFFICIENT_EVIDENCE"]:
                event.event_type = "INSUFFICIENT_EVIDENCE"
                event.details += " (Insufficient sustained duration)"

            event.status = "CLOSED"
            self.completed_events.append(event)

    def finalize(self) -> List[EventRecord]:
        """Flushes all remaining active events at end of video stream."""
        for key, event in list(self.active_events.items()):
            duration = event.end_ts - event.start_ts
            if duration < 0.35 and event.event_type not in ["EXEMPT_WALKING", "INSUFFICIENT_EVIDENCE"]:
                event.event_type = "INSUFFICIENT_EVIDENCE"
            event.status = "CLOSED"
            self.completed_events.append(event)
        self.active_events.clear()
        self.last_seen_frame_for_event.clear()
        return self.completed_events

    def export_json(self, output_path: Path) -> List[Dict[str, Any]]:
        """Exports all finalized events to JSON matching the evaluation schema."""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        records = [ev.to_dict() for ev in self.completed_events]
        payload = {
            "version": "1.0",
            "total_events": len(records),
            "events": records
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        return records

    def evaluate_against_ground_truth(self, gt_path: Path) -> Dict[str, Any]:
        """
        Evaluates detected events against ground truth JSON dataset.
        Computes TP, FP, FN, Precision, Recall, and F1 per event category.
        """
        if not gt_path.exists():
            return {"error": f"Ground truth not found at {gt_path}"}

        with open(gt_path, "r", encoding="utf-8") as f:
            gt_data = json.load(f)

        gt_events = gt_data.get("events", [])
        pred_events = [ev.to_dict() for ev in self.completed_events]

        metrics = {
            "total_gt": len(gt_events),
            "total_pred": len(pred_events),
            "categories": defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0}),
            "matches": []
        }

        matched_gt = set()
        matched_pred = set()

        for p_idx, pred in enumerate(pred_events):
            p_tid = pred["track_id"]
            p_type = pred["event_type"]
            p_start = pred["start_ts"]
            p_end = pred["end_ts"]

            best_gt_idx = None
            for g_idx, gt in enumerate(gt_events):
                if g_idx in matched_gt:
                    continue
                g_tid = gt["track_id"]
                g_type = gt["event_type"]
                g_start = gt["start_ts"]
                g_end = gt["end_ts"]

                time_overlap = max(0, min(p_end, g_end) - max(p_start, g_start))
                is_match = (p_tid == g_tid) or (time_overlap > 0.4 and p_type == g_type)

                if is_match and p_type == g_type:
                    best_gt_idx = g_idx
                    break

            if best_gt_idx is not None:
                matched_gt.add(best_gt_idx)
                matched_pred.add(p_idx)
                metrics["categories"][p_type]["TP"] += 1
                metrics["matches"].append({
                    "gt_id": gt_events[best_gt_idx]["id"],
                    "pred_id": pred["id"],
                    "event_type": p_type,
                    "status": "TP"
                })
            else:
                metrics["categories"][p_type]["FP"] += 1

        for g_idx, gt in enumerate(gt_events):
            if g_idx not in matched_gt:
                g_type = gt["event_type"]
                metrics["categories"][g_type]["FN"] += 1

        total_tp = sum(c["TP"] for c in metrics["categories"].values())
        total_fp = sum(c["FP"] for c in metrics["categories"].values())
        total_fn = sum(c["FN"] for c in metrics["categories"].values())

        prec = total_tp / max(1, total_tp + total_fp)
        rec = total_tp / max(1, total_tp + total_fn)
        f1 = 2 * prec * rec / max(1e-6, prec + rec)

        metrics["overall"] = {
            "precision": round(prec, 3),
            "recall": round(rec, 3),
            "f1_score": round(f1, 3)
        }
        return metrics


## 12. Visual Explanation & Trajectory Evidence Generator (`src/analytics/explanation.py`)
Saves comprehensive multi-panel incident cards with trajectory breadcrumbs, vehicle crop, and decision metrics.


In [ ]:
"""
Explanation Engine: Generates Supporting Video Evidence Frames & Trajectory Visualizations
Produces high-resolution explanation cards for each detected safety event.
"""
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
import cv2
import numpy as np

class ExplanationEngine:
    def __init__(self, output_dir: Path):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def generate_event_evidence(
        self,
        event: Any,
        peak_frame: np.ndarray,
        trajectory: List[Tuple[float, float]],
        bbox: Optional[np.ndarray] = None,
        crop_image: Optional[np.ndarray] = None,
        rationale: str = ""
    ) -> Path:
        """
        Creates a comprehensive multi-panel visual audit card for a reported event.
        Includes peak frame with trajectory breadcrumbs, crop of the vehicle, and decision metrics.
        """
        h, w = peak_frame.shape[:2]
        canvas = peak_frame.copy()

        # 1. Draw Trajectory Breadcrumbs
        if len(trajectory) >= 2:
            pts = np.array(trajectory, dtype=np.int32).reshape((-1, 1, 2))
            # Fading line
            for i in range(len(pts) - 1):
                pt1 = tuple(pts[i][0])
                pt2 = tuple(pts[i + 1][0])
                alpha = (i + 1) / float(len(pts))
                color = self._get_event_bgr_color(event.event_type)
                thickness = max(2, int(4 * alpha))
                cv2.line(canvas, pt1, pt2, color, thickness, cv2.LINE_AA)
                cv2.circle(canvas, pt2, max(3, int(6 * alpha)), color, -1)

            # Draw start point (Green circle) and latest point
            cv2.circle(canvas, tuple(pts[0][0]), 8, (0, 255, 0), -1)
            cv2.putText(canvas, "START", (pts[0][0][0] + 12, pts[0][0][1] + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # 2. Draw Target Bounding Box
        if bbox is not None:
            x1, y1, x2, y2 = map(int, bbox)
            box_color = self._get_event_bgr_color(event.event_type)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), box_color, 4)
            badge_text = f"#{int(event.track_id)} {event.event_type}"
            (tw, th), _ = cv2.getTextSize(badge_text, cv2.FONT_HERSHEY_SIMPLEX, 0.75, 2)
            cv2.rectangle(canvas, (x1, max(0, y1 - th - 10)), (x1 + tw + 10, y1), box_color, -1)
            cv2.putText(canvas, badge_text, (x1 + 5, max(th + 5, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

        # 3. Add High-Level HUD Explanation Banner
        banner_h = 130
        banner = np.zeros((banner_h, w, 3), dtype=np.uint8)
        banner[:] = (24, 24, 24)

        title = f"EVENT EVIDENCE REPORT: {event.id} [{event.event_type}] - TRACK #{int(event.track_id)}"
        timing = f"Time: {float(event.start_ts):.2f}s - {float(event.end_ts):.2f}s (Duration: {float(event.end_ts - event.start_ts):.2f}s) | Conf: {float(event.confidence):.2f}"
        metrics = f"Zone: {getattr(event, 'lane_or_zone', 'N/A')} | Rationale: {rationale or getattr(event, 'details', 'N/A')}"

        cv2.putText(banner, title, (25, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.95, self._get_event_bgr_color(event.event_type), 2, cv2.LINE_AA)
        cv2.putText(banner, timing, (25, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.70, (220, 220, 220), 2, cv2.LINE_AA)
        cv2.putText(banner, metrics, (25, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (180, 240, 255), 2, cv2.LINE_AA)

        # Compose full evidence card
        full_card = np.vstack([banner, canvas])

        # Save to disk
        filename = f"{event.id}_{event.event_type}_tid{int(event.track_id)}.jpg"
        out_path = self.output_dir / filename
        cv2.imwrite(str(out_path), full_card)
        return out_path

    def _get_event_bgr_color(self, event_type: str) -> Tuple[int, int, int]:
        if event_type == "WRONG_WAY":
            return (0, 0, 255)       # Red
        elif event_type == "CROSSWALK_RIDING":
            return (0, 140, 255)     # Orange
        elif event_type == "EXEMPT_WALKING":
            return (255, 220, 0)     # Cyan
        elif event_type == "INSUFFICIENT_EVIDENCE":
            return (0, 215, 255)     # Amber/Yellow
        return (0, 255, 0)           # Green



## 13. Integrated Intersection Safety Pipeline (`src/pipeline.py`)
Orchestrates detection, tracking, safety checks, event verification, and visual evidence generation.


In [ ]:
"""
Complete End-to-End CCTV Intersection Safety & Motorcycle Violation Pipeline
Orchestrates Detection, Tracking, Trajectory Analysis, Wrong-Way Detection,
Crosswalk Riding / Exemption Verification, Event Lifecycle, and Visual Evidence Generation.
"""
from collections import defaultdict, deque
import json
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
import torch

from src.config import load_intersection_config, IntersectionConfig
from src.core import get_box_geometry, get_ground_anchor, draw_render_progress_bar
from src.tracking import RobustTrackerManager
from src.safety import (
    AutoLaneDetector,
    DirectionChecker,
    CrosswalkSafetyChecker,
    classify_vehicle_type,
    resolve_riders_and_pedestrians,
    PedestrianSafetyChecker,
)
from src.analytics import EventVerifier, ExplanationEngine

class IntersectionSafetyPipeline:
    def __init__(
        self,
        config_path: Path,
        model_path: Optional[Path] = None,
        output_dir: Optional[Path] = None,
        device: Optional[str] = None,
        imgsz: int = 960
    ):
        self.config_path = Path(config_path)
        self.config = load_intersection_config(self.config_path)
        self.imgsz = imgsz

        if output_dir is None:
            self.output_dir = self.config_path.parent.parent / "data" / "output"
        else:
            self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.events_dir = self.output_dir / "events"
        self.events_dir.mkdir(parents=True, exist_ok=True)

        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        # Prefer YOLO11m (Medium) if available for highest accuracy on occluded traffic & small bikes
        if model_path is None:
            yolo11m_path = self.config_path.parent.parent / "models" / "yolo11m.pt"
            yolo11s_path = self.config_path.parent.parent / "models" / "yolo11s.pt"
            if yolo11m_path.exists():
                model_path = yolo11m_path
            elif yolo11s_path.exists():
                model_path = yolo11s_path
            else:
                model_path = self.config_path.parent.parent / "models" / "yolov8s.pt"

        print(f"Loading Detection Model from: {model_path} on {self.device.upper()} (imgsz={self.imgsz})...")
        self.model = YOLO(str(model_path))
        self.model.to(self.device)

        # 1. Tracker Manager (Item 1)
        self.tracker = RobustTrackerManager(
            max_reconnect_frames=25,
            max_reconnect_dist_px=180.0,
            perspective_cfg=self.config.perspective_scale_by_y
        )

        # 2. Lane Detector & Ambiguity Handler (Item 2)
        self.lane_detector = AutoLaneDetector(
            road_polygon=np.array(self.config.road.polygon, dtype=np.int32),
            warmup_frames=self.config.road.warmup_frames,
            perspective_cfg=self.config.perspective_scale_by_y
        )
        self.lane_detector.calibrate(fallback_lanes=self.config.lanes)

        # 3. Direction Checker (Item 3)
        self.direction_checker = DirectionChecker(
            min_frames=5,
            min_sustained_frames=5,
            min_recent_speed=2.0,
            min_total_dist_px=20.0,
            wrong_way_threshold=-0.60,
            perspective_cfg=self.config.perspective_scale_by_y
        )

        # 4. Crosswalk Riding vs Walking Checker (Item 4)
        cw_polys = [np.array(cw.polygon, dtype=np.int32) for cw in self.config.crosswalks]
        self.crosswalk_checker = CrosswalkSafetyChecker(
            crosswalk_polygons=cw_polys,
            min_passage_dist_px=40.0,
            max_walking_speed=3.8,
            min_riding_speed=4.8,
            perspective_cfg=self.config.perspective_scale_by_y
        )

        # 5. Pedestrian Safety Checker
        self.ped_safety = PedestrianSafetyChecker(min_confirm_frames=4)

        # 6. Event Verifier & Lifecycle Manager (Item 5)
        self.event_verifier = EventVerifier(fps=self.config.fps)

        # 7. Explanation Engine (Item 6)
        self.explanation_engine = ExplanationEngine(output_dir=self.events_dir)

        # Visual Annotators
        self._init_annotators()

        # Cache of peak frames and trajectories for event explanation
        self.track_peak_frames: Dict[int, np.ndarray] = {}
        self.track_peak_bboxes: Dict[int, np.ndarray] = {}

    def _init_annotators(self):
        self.LANE_PALETTE = sv.ColorPalette([
            sv.Color.from_hex("#00C853"),  # Green
            sv.Color.from_hex("#2979FF"),  # Blue
        ])
        self.CW_COLOR = sv.Color.from_hex("#00E5FF")    # Cyan
        self.ALERT_COLOR = sv.Color.RED                # Red
        self.WARN_COLOR = sv.Color.from_hex("#FFA000") # Orange

        self.lane_annotators = [
            sv.PolygonZoneAnnotator(zone=z, color=self.LANE_PALETTE.by_idx(i), thickness=3)
            for i, z in enumerate(self.lane_detector.lane_zones)
        ]
        self.cw_annotators = [
            sv.PolygonZoneAnnotator(zone=z, color=self.CW_COLOR, thickness=3)
            for z in self.crosswalk_checker.crosswalk_zones
        ]
        self.normal_box = sv.BoxAnnotator(color=sv.Color.GREEN, thickness=2)
        self.normal_label = sv.LabelAnnotator(
            color=sv.Color.GREEN, text_color=sv.Color.WHITE, text_scale=0.55, text_padding=3
        )

        self.alert_box = sv.BoxAnnotator(color=self.ALERT_COLOR, thickness=4)
        self.alert_label = sv.LabelAnnotator(
            color=self.ALERT_COLOR, text_color=sv.Color.WHITE, text_scale=0.6, text_padding=3
        )

        self.warn_box = sv.BoxAnnotator(color=self.WARN_COLOR, thickness=3)
        self.warn_label = sv.LabelAnnotator(
            color=self.WARN_COLOR, text_color=sv.Color.BLACK, text_scale=0.55, text_padding=3
        )

        self.cyan_box = sv.BoxAnnotator(color=self.CW_COLOR, thickness=3)
        self.cyan_label = sv.LabelAnnotator(
            color=self.CW_COLOR, text_color=sv.Color.BLACK, text_scale=0.55, text_padding=3
        )

    def process_frame(
        self,
        frame: np.ndarray,
        frame_idx: int,
        total_frames: int = 0
    ) -> np.ndarray:
        """Processes a single video frame through the complete safety pipeline."""
        annotated_frame = frame.copy()

        # 1. Detection and Tracking with YOLO11 / YOLOv8 using ByteTrack
        results = self.model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            classes=[0, 1, 2, 3, 5, 7],
            imgsz=self.imgsz,
            device=self.device,
            conf=0.20,
            verbose=False
        )[0]
        detections = sv.Detections.from_ultralytics(results)

        # 2. Filter out ALL objects outside the monitored road and crosswalk zones
        # Protects against annotating parked vehicles, off-road tuk-tuks, or background bystanders
        if len(detections) > 0:
            inside_zone_mask = np.zeros(len(detections), dtype=bool)
            road_poly = np.array(self.config.road.polygon, dtype=np.int32)
            for idx, xyxy in enumerate(detections.xyxy):
                cx, cy = get_ground_anchor(xyxy)
                pt = (float(cx), float(cy))
                # cv2.pointPolygonTest returns +1 (inside), 0 (on edge), -1 (outside)
                if cv2.pointPolygonTest(road_poly, pt, False) >= 0:
                    inside_zone_mask[idx] = True
                else:
                    for cw_poly in self.crosswalk_checker.crosswalk_polygons:
                        if cv2.pointPolygonTest(cw_poly, pt, False) >= 0:
                            inside_zone_mask[idx] = True
                            break
            detections = detections[inside_zone_mask]

        # Assign fallback tracker IDs if running isolated frames or tracker warmup
        if detections.tracker_id is None or len(detections.tracker_id) == 0:
            detections.tracker_id = np.arange(1, len(detections) + 1, dtype=int)
        else:
            tids = list(detections.tracker_id)
            for i in range(len(tids)):
                if tids[i] is None or tids[i] <= 0:
                    tids[i] = i + 1
            detections.tracker_id = np.array(tids, dtype=int)

        # 3. Suppress duplicate bike boxes first
        detections = self.tracker.suppress_duplicate_bikes(detections, iou_threshold=0.25)

        # 4. Build track history lookup before rider resolution
        raw_track_hist = {
            cid: [(p[1], p[2]) for p in trk.history]
            for cid, trk in self.tracker.tracks.items()
        }

        # 5. Rider-to-Bike Fusion: merge mounted riders into bikes; keep genuine pedestrians
        detections = resolve_riders_and_pedestrians(
            detections=detections,
            lane_zones=self.lane_detector.lane_zones,
            crosswalk_zones=self.crosswalk_checker.crosswalk_zones,
            track_history=raw_track_hist
        )

        # 6. Update track histories & resolve ID switching
        detections = self.tracker.update_tracks(detections, frame_idx)

        # Re-sync raw track history with newly updated tracks
        raw_track_hist = {
            cid: [(p[1], p[2]) for p in trk.history]
            for cid, trk in self.tracker.tracks.items()
        }

        # 7. Draw Crosswalk and Lane Polygons
        for cw_ann in self.cw_annotators:
            annotated_frame = cw_ann.annotate(scene=annotated_frame)
        for lane_ann in self.lane_annotators:
            annotated_frame = lane_ann.annotate(scene=annotated_frame)

        # 8. Check Genuine Pedestrian Safety on Foot
        # Hazard: Pedestrians walking/crossing active traffic lanes outside crosswalk (ALERT RED)
        # Safe: Pedestrians walking inside designated crosswalk (CYAN)
        hazard_peds, safe_peds = self.ped_safety.check_safety(
            detections, self.lane_detector.lane_zones, self.crosswalk_checker.crosswalk_zones, raw_track_hist, frame_idx
        )
        hazard_tids = set(hazard_peds.tracker_id) if len(hazard_peds) > 0 and hazard_peds.tracker_id is not None else set()
        safe_tids = set(safe_peds.tracker_id) if len(safe_peds) > 0 and safe_peds.tracker_id is not None else set()

        if len(safe_peds) > 0:
            annotated_frame = self.cyan_box.annotate(scene=annotated_frame, detections=safe_peds)
            labels = [f"#{tid} Ped (Safe)" for tid in safe_peds.tracker_id]
            annotated_frame = self.cyan_label.annotate(scene=annotated_frame, detections=safe_peds, labels=labels)

        if len(hazard_peds) > 0:
            annotated_frame = self.alert_box.annotate(scene=annotated_frame, detections=hazard_peds)
            labels = [f"[!] #{tid} Ped in Lane" for tid in hazard_peds.tracker_id]
            annotated_frame = self.alert_label.annotate(scene=annotated_frame, detections=hazard_peds, labels=labels)

        # 9. Check Crosswalk Traversal (Riding vs Walking a Motorcycle)
        # Normal motorcycles traveling down their traffic lane in permitted direction are exempted
        lane_vectors = [lane.permitted_unit_vector for lane in self.config.lanes]
        cw_results = self.crosswalk_checker.check_motorcycles(
            detections=detections,
            track_history=raw_track_hist,
            frame_idx=frame_idx,
            lane_vectors=lane_vectors
        )

        for tid, cw_status in cw_results.items():
            if cw_status == "CROSSWALK_RIDING":
                self.event_verifier.log_candidate(
                    track_id=tid,
                    event_type="CROSSWALK_RIDING",
                    frame_idx=frame_idx,
                    lane_or_zone="crosswalk",
                    details="Motorcycle traversed crosswalk above walking speed",
                    track_len=len(self.tracker.tracks[tid].history) if tid in self.tracker.tracks else 10
                )
                self.track_peak_frames[tid] = frame.copy()
            elif cw_status == "EXEMPT_WALKING":
                self.event_verifier.log_candidate(
                    track_id=tid,
                    event_type="EXEMPT_WALKING",
                    frame_idx=frame_idx,
                    lane_or_zone="crosswalk",
                    details="Rider dismounted walking motorcycle across crosswalk",
                    track_len=len(self.tracker.tracks[tid].history) if tid in self.tracker.tracks else 10
                )
                self.track_peak_frames[tid] = frame.copy()

        # 9. Process ALL Vehicles Across the Entire Road (No Unannotated Vehicles!)
        vehicle_dets = detections[detections.class_id != 0]
        if len(vehicle_dets) > 0:
            # Check lane assignments and lane divider ambiguity
            ambiguous_mask = np.zeros(len(vehicle_dets), dtype=bool)
            lane_assignment = [None] * len(vehicle_dets)

            for idx, xyxy in enumerate(vehicle_dets.xyxy):
                cx, cy = get_ground_anchor(xyxy)
                lane_idx, is_amb, _ = self.lane_detector.assign_lane((cx, cy))
                ambiguous_mask[idx] = is_amb
                lane_assignment[idx] = lane_idx

            # Track which vehicles are confirmed wrong-way
            confirmed_wrong_tids = set()

            for lane_idx, lane in enumerate(self.config.lanes):
                in_this_lane = np.array([l == lane_idx for l in lane_assignment], dtype=bool)
                if not np.any(in_this_lane):
                    continue

                lane_dets = vehicle_dets[in_this_lane]
                lane_amb_mask = ambiguous_mask[in_this_lane]

                confirmed_wrong, suspected_wrong = self.direction_checker.check_detections(
                    detections=lane_dets,
                    permitted_vector=lane.permitted_unit_vector,
                    track_history=raw_track_hist,
                    lane_id=lane.id,
                    frame_idx=frame_idx,
                    ambiguous_mask=lane_amb_mask
                )

                for idx, (tid, is_conf, is_susp) in enumerate(zip(lane_dets.tracker_id, confirmed_wrong, suspected_wrong)):
                    track_len = len(self.tracker.tracks[tid].history) if tid in self.tracker.tracks else 10
                    is_ambiguous = lane_amb_mask[idx]

                    if is_conf:
                        confirmed_wrong_tids.add(tid)
                        self.event_verifier.log_candidate(
                            track_id=tid,
                            event_type="WRONG_WAY",
                            frame_idx=frame_idx,
                            lane_or_zone=lane.id,
                            details=f"Sustained wrong-way movement counter to {lane.permitted_unit_vector}",
                            is_ambiguous=is_ambiguous,
                            track_len=track_len
                        )
                        self.track_peak_frames[tid] = frame.copy()
                        self.track_peak_bboxes[tid] = lane_dets.xyxy[idx].copy()
                    elif is_susp and (is_ambiguous or track_len < 6):
                        self.event_verifier.log_candidate(
                            track_id=tid,
                            event_type="INSUFFICIENT_EVIDENCE",
                            frame_idx=frame_idx,
                            lane_or_zone=lane.id,
                            details="Brief counter-flow deviation with ambiguous lane boundary or short track",
                            is_ambiguous=True,
                            track_len=track_len
                        )

            # Separate all vehicles into categories for clean annotation
            wrong_mask = np.isin(vehicle_dets.tracker_id, list(confirmed_wrong_tids))
            cw_riding_mask = np.array([cw_results.get(tid) == "CROSSWALK_RIDING" for tid in vehicle_dets.tracker_id], dtype=bool)
            cw_walking_mask = np.array([cw_results.get(tid) == "EXEMPT_WALKING" for tid in vehicle_dets.tracker_id], dtype=bool)

            # 9A. Confirmed Wrong-Way Vehicles (Alert Red)
            if np.any(wrong_mask):
                wrong_dets = vehicle_dets[wrong_mask]
                annotated_frame = self.alert_box.annotate(scene=annotated_frame, detections=wrong_dets)
                labels = [f"[!] #{tid} Wrong Way" for tid in wrong_dets.tracker_id]
                annotated_frame = self.alert_label.annotate(scene=annotated_frame, detections=wrong_dets, labels=labels)

            # 9B. Crosswalk Riding Violations (Alert Orange)
            if np.any(cw_riding_mask & ~wrong_mask):
                riding_dets = vehicle_dets[cw_riding_mask & ~wrong_mask]
                annotated_frame = self.warn_box.annotate(scene=annotated_frame, detections=riding_dets)
                labels = [f"[!] #{tid} CW Riding" for tid in riding_dets.tracker_id]
                annotated_frame = self.warn_label.annotate(scene=annotated_frame, detections=riding_dets, labels=labels)

            # 9C. Crosswalk Walking Exemption (Cyan)
            if np.any(cw_walking_mask & ~wrong_mask & ~cw_riding_mask):
                walking_dets = vehicle_dets[cw_walking_mask & ~wrong_mask & ~cw_riding_mask]
                annotated_frame = self.cyan_box.annotate(scene=annotated_frame, detections=walking_dets)
                labels = [f"#{tid} Walking (Exempt)" for tid in walking_dets.tracker_id]
                annotated_frame = self.cyan_label.annotate(scene=annotated_frame, detections=walking_dets, labels=labels)

            # 9D. All Normal Vehicles Everywhere on Road (Green)
            normal_mask = ~(wrong_mask | cw_riding_mask | cw_walking_mask)
            if np.any(normal_mask):
                normal_dets = vehicle_dets[normal_mask]
                annotated_frame = self.normal_box.annotate(scene=annotated_frame, detections=normal_dets)
                labels = [
                    f"#{tid} {classify_vehicle_type(cid, box)}"
                    for cid, tid, box in zip(normal_dets.class_id, normal_dets.tracker_id, normal_dets.xyxy)
                ]
                annotated_frame = self.normal_label.annotate(scene=annotated_frame, detections=normal_dets, labels=labels)

        # 10. Step Event Lifecycle
        self.event_verifier.step(frame_idx)

        # 11. Render HUD Progress Bar
        if total_frames > 0:
            annotated_frame = draw_render_progress_bar(annotated_frame, frame_idx + 1, total_frames)

        return annotated_frame

    def finalize(self) -> Tuple[List[Any], Dict[str, Any]]:
        """
        Finalizes event logging, exports detected_events.json,
        generates supporting explanation images, and evaluates against eval_gt.json.
        """
        print("\nFinalizing safety event records...")
        final_events = self.event_verifier.finalize()

        # Export events JSON
        events_json_path = self.output_dir / "detected_events.json"
        self.event_verifier.export_json(events_json_path)
        print(f"Exported {len(final_events)} verified events to: {events_json_path}")

        # Generate visual evidence frames for each event
        print(f"Generating visual audit evidence frames in {self.events_dir}...")
        for event in final_events:
            tid = event.track_id
            peak_frame = self.track_peak_frames.get(tid)
            trajectory = []
            if tid in self.tracker.tracks:
                trajectory = [(p[1], p[2]) for p in self.tracker.tracks[tid].history]

            bbox = self.track_peak_bboxes.get(tid)
            if bbox is None and tid in self.tracker.tracks and len(self.tracker.tracks[tid].history) > 0:
                bbox = self.tracker.tracks[tid].history[-1][3]

            if peak_frame is not None and len(trajectory) > 0:
                self.explanation_engine.generate_event_evidence(
                    event=event,
                    peak_frame=peak_frame,
                    trajectory=trajectory,
                    bbox=bbox,
                    rationale=event.details
                )

        # Evaluate against Ground Truth
        gt_path = self.config_path.parent.parent / "data" / "eval_gt.json"
        eval_metrics = self.event_verifier.evaluate_against_ground_truth(gt_path)

        eval_summary_path = self.output_dir / "evaluation_summary.json"
        with open(eval_summary_path, "w", encoding="utf-8") as f:
            json.dump(eval_metrics, f, indent=2)
        print(f"Saved benchmark evaluation metrics to: {eval_summary_path}")

        return (final_events, eval_metrics)


## 14. Full Video Stream Processing & Benchmark Evaluation
Runs the complete pipeline on `intersection_01.mp4`, outputs annotated video with HUD progress bar, exports `detected_events.json`, creates evidence cards in `data/output/events/`, and prints benchmark evaluation scores.


In [ ]:
# Execute End-to-End Safety Pipeline on Intersection Video
from pathlib import Path
import supervision as sv
from src.pipeline import IntersectionSafetyPipeline

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent

CONFIG_PATH = BASE_DIR / "configs" / "zones.json"
MODEL_PATH = BASE_DIR / "models" / "yolo11m.pt"
if not MODEL_PATH.exists():
    MODEL_PATH = BASE_DIR / "models" / "yolov8s.pt"

VIDEO_PATH = BASE_DIR / "data" / "videos" / "intersection_01.mp4"
OUTPUT_DIR = BASE_DIR / "data" / "output"
TARGET_VIDEO = OUTPUT_DIR / "intersection_01_result.mp4"

pipeline = IntersectionSafetyPipeline(
    config_path=CONFIG_PATH,
    model_path=MODEL_PATH,
    output_dir=OUTPUT_DIR
)

video_info = sv.VideoInfo.from_video_path(str(VIDEO_PATH))
total = video_info.total_frames
print(f"Processing Video Stream: {VIDEO_PATH.name} ({total} frames @ {video_info.fps} FPS)")

frame_generator = sv.get_video_frames_generator(source_path=str(VIDEO_PATH))

with sv.VideoSink(target_path=str(TARGET_VIDEO), video_info=video_info) as sink:
    for frame_idx, frame in enumerate(frame_generator):
        annotated_frame = pipeline.process_frame(frame, frame_idx, total_frames=total)
        sink.write_frame(annotated_frame)
        if frame_idx % 25 == 0 or frame_idx == total - 1:
            pct = ((frame_idx + 1) / total) * 100.0
            print(f"\rProgress: Frame {frame_idx + 1:04d}/{total:04d} ({pct:5.1f}%)", end="", flush=True)

print("\nProcessing complete! Finalizing events and benchmark evaluation...")
final_events, metrics = pipeline.finalize()

print(f"\n--- BENCHMARK EVALUATION SUMMARY (Overall F1: {metrics.get('overall', {}).get('f1_score', 'N/A')}) ---")
print(f"Total Ground Truth Events: {metrics.get('total_gt')}")
print(f"Total Detected Events: {metrics.get('total_pred')}")
for cat, stats in metrics.get('categories', {}).items():
    print(f" - {cat}: TP={stats['TP']}, FP={stats['FP']}, FN={stats['FN']}")


## 15. Random Sample Frame Visual Verification
Inspects sample frames across random timestamps to audit detection, vehicle classification, tracking, and violation labeling quality.


In [ ]:
# Verify Random Frames Across Video Timeline with Warmup
import cv2
from pathlib import Path
from src.pipeline import IntersectionSafetyPipeline

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent

CONFIG_PATH = BASE_DIR / "configs" / "zones.json"
MODEL_PATH = BASE_DIR / "models" / "yolo11m.pt"
if not MODEL_PATH.exists():
    MODEL_PATH = BASE_DIR / "models" / "yolov8s.pt"

VIDEO_PATH = BASE_DIR / "data" / "videos" / "intersection_01.mp4"
VERIFY_DIR = BASE_DIR / "data" / "output" / "random_frames_verification"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

pipeline = IntersectionSafetyPipeline(
    config_path=CONFIG_PATH,
    model_path=MODEL_PATH,
    imgsz=960
)

# Inspect sample frames across different time intervals: 0.5s, 2.2s, 4.7s, 9.3s
target_frames = [15, 65, 140, 280]

print("Running random frame verification with 12-frame tracking warmup...")
for target in target_frames:
    start_f = max(0, target - 12)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
    for f in range(start_f, target + 1):
        ret, frame = cap.read()
        if not ret:
            break
        ann = pipeline.process_frame(frame, frame_idx=f, total_frames=total)
        if f == target:
            t_s = target / fps
            out_file = VERIFY_DIR / f"verified_frame_{target:04d}_time_{t_s:04.1f}s.jpg"
            cv2.imwrite(str(out_file), ann)
            print(f"Verified & Saved: {out_file.name}")

cap.release()
print("Random frame verification audit complete!")
